## HDF5 Inspection

### Setup

In [9]:
import h5py
import numpy as np
import pandas as pd

from _notebook_setup import RAW_DIR, require_path

raw_path = require_path(RAW_DIR / "Halfmile3D_add_geom_sorted.hdf5")
hdf5_group_name = "TRACE_DATA/DEFAULT"

raw_path

WindowsPath('D:/github_repos/seismic-first-break-picker/data/raw/Halfmile3D_add_geom_sorted.hdf5')

## Available HDF5 Fields

In [10]:
with h5py.File(raw_path, "r") as handle:
    group = handle[hdf5_group_name]
    field_rows = []
    for key in sorted(group.keys()):
        dataset = group[key]
        field_rows.append(
            {
                "field": key,
                "shape": tuple(dataset.shape),
                "dtype": str(dataset.dtype),
            }
        )

fields_df = pd.DataFrame(field_rows)
fields_df

,field,shape,dtype
0,ALIAS_FREQ,"(1099559, 1)",uint32
1,ALIAS_SLOPE,"(1099559, 1)",int32
2,AZIMUTH,"(1099559, 1)",int32
3,CDP,"(1099559, 1)",uint32
4,CDPTRACE,"(1099559, 1)",uint32
...,...,...,...
99,data_array,"(1099559, 751)",float32
100,processing_history,"(16, 1)",|S256
101,reel_headers,"(1,)","[('JOB', '<u4'), ('LINE', '<u4'), ('REEL', '<u..."
102,reel_text_header,"(1, 1)",|S3200


## Core Dataset Summary

The next cell checks the fields that matter most for this task: the trace array, shot identifier, sampling interval, and `SPARE1` first-break labels. Values `0` and `-1` are treated as unlabeled.

In [11]:
with h5py.File(raw_path, "r") as handle:
    group = handle[hdf5_group_name]

    data_shape = group["data_array"].shape
    labels_ms = np.asarray(group["SPARE1"]).reshape(-1)
    sample_rate_us = np.asarray(group["SAMP_RATE"]).reshape(-1)
    sample_num = np.asarray(group["SAMP_NUM"]).reshape(-1)

    shot_key = "SHOTID" if "SHOTID" in group else "SHOT_PEG"
    shot = np.asarray(group[shot_key]).reshape(-1)

    valid_label_mask = labels_ms > 0

summary_df = pd.DataFrame(
    {
        "metric": [
            "hdf5_group",
            "trace_count",
            "sample_count_per_trace",
            "sample_interval_ms",
            "shot_key",
            "unique_shots",
            "valid_labeled_traces",
            "label_coverage_percent",
            "sample_num_unique_count",
        ],
        "value": [
            hdf5_group_name,
            data_shape[0],
            data_shape[1],
            float(sample_rate_us[0]) / 1000.0,
            shot_key,
            int(np.unique(shot).size),
            int(valid_label_mask.sum()),
            round(float(valid_label_mask.mean() * 100.0), 2),
            int(np.unique(sample_num).size),
        ],
    }
)

summary_df

,metric,value
0,hdf5_group,TRACE_DATA/DEFAULT
1,trace_count,1099559
2,sample_count_per_trace,751
3,sample_interval_ms,2.0
4,shot_key,SHOTID
5,unique_shots,690
6,valid_labeled_traces,993189
7,label_coverage_percent,90.33
8,sample_num_unique_count,1


### Label Distribution Quick Check

This is a lightweight check of the manual first-break labels. It confirms that the notebook is using positive `SPARE1` values as valid labels and ignoring unlabeled traces.

In [12]:
valid_labels_ms = labels_ms[valid_label_mask]

label_distribution_df = pd.DataFrame(
    {
        "metric": ["min_ms", "p25_ms", "median_ms", "p75_ms", "max_ms"],
        "value": [
            float(np.min(valid_labels_ms)),
            float(np.percentile(valid_labels_ms, 25)),
            float(np.median(valid_labels_ms)),
            float(np.percentile(valid_labels_ms, 75)),
            float(np.max(valid_labels_ms)),
        ],
    }
)

label_distribution_df

,metric,value
0,min_ms,11.0
1,p25_ms,238.0
2,median_ms,344.0
3,p75_ms,454.0
4,max_ms,1482.0


## What this confirms

At this point we have confirmed that the raw file is readable, the required task group exists, the manual first-break labels are present in `SPARE1`, and the dataset has enough labeled traces to continue with 2D panel reconstruction. The next pipeline step is segment export: grouping traces by shot and ordering them by receiver geometry.